# AML GNN — Enhanced Data Preparation  *(v2)*

Drop-in replacement for `Data_prepration.ipynb`.
Keeps all original features (Section 1) and adds fully symmetric temporal
account-state features (Section 2) attached to each edge.

## Feature inventory (v2)

| Section | Group | Cols | Key AML signal |
|---|---|---|---|
| 1 | Transaction-level baseline (v2) | 9 | FX-corrected amount, payment-format risk score, timing |
| 2.1 | Sender — outgoing history | 9 | Sending bursts, fan-out, structuring |
| 2.2 | Sender — incoming history | 9 | Upstream funds, pass-through detection |
| 2.3 | Receiver — incoming history | 9 | Fan-in accumulation, mule detection |
| 2.4 | Receiver — outgoing history | 9 | Downstream scatter, pass-through detection |
| 2.5 | Scatter-gather (6 h, both sides) | 8 | Fan-In/Fan-Out burst, Gather-Scatter |
| 2.6 | Cycle / structural (both accounts) | 4 | Circular flows, loop participation |
| | **Grand total (edge_attr dim)** | **57** | |

## Node feature inventory (v2)

| Feature | Cols | Key signal |
|---|---|---|
| `Bank_TargetEnc` | 1 | Smoothed laundering rate per bank (training edges only) |
| `EntityType_TargetEnc` | 1 | Smoothed laundering rate per entity type (training edges only) |
| | **2** | |

## Symmetry principle

Every account involved in a transaction has **four behavioural dimensions**:

```
u (sender)  :  src_out_*  (what u sends)    +  src_in_*  (what u receives)
v (receiver):  dst_in_*   (what v receives) +  dst_out_* (what v sends)
```

Without `src_in_*` the model cannot see that the sender was itself a fan-in
target moments before forwarding funds.  Without `dst_out_*` it cannot detect
that the receiver immediately re-scatters what it receives.
Cycle participation is tracked for both accounts in Section 2.6.

## Temporal correctness guarantee

Every feature for edge `(u→v, t)` uses **only transactions with timestamp < t**.
Enforced by:
- sorting the full dataframe by `Timestamp` once at the start,
- `.shift(1)` (position-based, cumulative) or `.shift(freq='1ms')`
  (time-index, rolling) to exclude the current row from its own history.


---
## 0. Imports & data loading

In [ ]:
import warnings, time
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
from collections import deque
import pickle

import torch
from torch_geometric.data import Data
from tqdm import tqdm

# ── Lightweight timer context manager ───────────────────────────────────────
# Usage:  with Timer('label'): ...  prints elapsed seconds on exit.
class Timer:
    def __init__(self, label): self.label = label
    def __enter__(self): self.t = time.time(); return self
    def __exit__(self, *a): print(f'     [{self.label}] done in {time.time()-self.t:.1f}s')

print('Libraries loaded ✓')

In [ ]:
df_tr = pd.read_csv('Data/LI-Small_Trans.csv', low_memory=False)
df_ac = pd.read_csv('Data/LI-Small_accounts.csv', low_memory=False)

print(f'Transactions : {len(df_tr):,}')
print(f'Accounts     : {len(df_ac):,}')
df_tr.head(3)

In [ ]:
import yfinance as yf

# Map payment currency names to Yahoo Finance FX tickers (currency vs USD)
CURRENCY_TO_ISO = {
    'US Dollar': 'USD', 'Euro': 'EUR', 'British Pound': 'GBP',
    'Australian Dollar': 'AUD', 'Canadian Dollar': 'CAD', 'Swiss Franc': 'CHF',
    'Chinese Yuan': 'CNY', 'Japanese Yen': 'JPY', 'Mexican Peso': 'MXN',
    'Brazilian Real': 'BRL', 'Indian Rupee': 'INR', 'South Korean Won': 'KRW',
    'Russian Ruble': 'RUB', 'Saudi Riyal': 'SAR', 'Singapore Dollar': 'SGD',
    'Hong Kong Dollar': 'HKD', 'Norwegian Krone': 'NOK', 'Swedish Krona': 'SEK',
    'Danish Krone': 'DKK', 'New Zealand Dollar': 'NZD', 'South African Rand': 'ZAR',
    'Turkish Lira': 'TRY', 'UAE Dirham': 'AED', 'Bitcoin': 'BTC', 'Ethereum': 'ETH',
}

_all_currencies = df_tr['Payment Currency'].unique()
_non_usd = [c for c in _all_currencies if c != 'US Dollar' and c in CURRENCY_TO_ISO]
print(f'Non-USD currencies in dataset: {_non_usd}')

# Fetch daily close rates for Sep 1-17 2022 + buffer
_START, _END = '2022-09-01', '2022-09-19'
# Full date range so weekends are forward-filled from last trading day
_full_dates = pd.date_range(start=_START, end='2022-09-17')

fx_rate_lookup = {}  # {(currency_name, date_str): rate_to_usd}

for _cname in _non_usd:
    _iso = CURRENCY_TO_ISO[_cname]
    # Crypto pairs use dash-USD format; FX pairs use ISO+USD=X format
    if _iso in ('BTC', 'ETH'):
        _ticker = f'{_iso}-USD'
    else:
        _ticker = f'{_iso}USD=X'
    try:
        _raw = yf.download(_ticker, start=_START, end=_END,
                           auto_adjust=True, progress=False)
        if _raw.empty:
            print(f'  WARNING: no data for {_ticker}, defaulting to 1.0')
            continue
        # Normalise to Series regardless of yfinance version
        _cl = _raw['Close']
        if hasattr(_cl, 'columns'):
            _cl = _cl.iloc[:, 0]
        _cl = _cl.squeeze()
        # Forward-fill weekends/holidays so every transaction date has a rate
        _cl = _cl.reindex(_full_dates).ffill().bfill()
        for _dt, _rate in _cl.items():
            if pd.notna(_rate):
                fx_rate_lookup[(_cname, str(_dt.date()))] = float(_rate)
        print(f'  {_cname} ({_ticker}): {_cl.notna().sum()} days covered, '
              f'last={float(_cl.iloc[-1]):.4f}')
    except Exception as _e:
        print(f'  ERROR fetching {_ticker}: {_e}')

print(f'\nFX lookup entries: {len(fx_rate_lookup):,}  (USD: rate=1.0 applied inline)')
print('FX rates loaded ✓')


---
## 1. Original transaction-level edge features  *(v2: FX-corrected amounts + target encodings)*

These 9 features describe **what happened in this transaction** — currency-normalised amount,
payment method risk, timing, and structure.  No historical account context is used.

**Changes from v1 baseline:**
- `Amount_Log` now converts each transaction to **USD using day-specific exchange rates**
  (fetched from Yahoo Finance in Section 0) before applying log1p, making amounts comparable
  across the 7 currencies present in this dataset.
- Payment Format OHE (7 cols) + `Is_ACH` (1 col) replaced by a single
  **training-set target encoding** (`PayFmt_TargetEnc`), which encodes the smoothed laundering
  rate per payment method.  Computed on training edges only (α = 20 smoothing) — no data leakage.

| Feature | How it is computed | AML intuition |
|---|---|---|
| `Amount_Log` | log1p(Amount Paid × FX_rate → USD) | Compresses the heavy-tailed amount distribution; FX normalisation makes cross-currency amounts directly comparable — a €1 000 wire and a $1 000 wire are now equivalent |
| `Currency_Mismatch` | 1 if Payment Currency ≠ Receiving Currency, else 0 | Laundering often routes through currency conversion to obscure the paper trail |
| `Hour_Sin` | sin(2π × hour / 24) | Cyclical encoding avoids the artificial discontinuity at midnight |
| `Hour_Cos` | cos(2π × hour / 24) | Paired with Hour_Sin to fully represent any time-of-day |
| `DayOfWeek_Sin` | sin(2π × day\_of\_week / 7) | Saturday/Sunday show ~2× the average laundering rate in this dataset |
| `DayOfWeek_Cos` | cos(2π × day\_of\_week / 7) | Paired with DayOfWeek_Sin to fully represent any day-of-week |
| `Is_Weekend` | 1 if day\_of\_week ∈ {5, 6}, else 0 | Explicit weekend flag alongside the cyclical encoding |
| `PayFmt_TargetEnc` | (Σ train labels + α × global\_rate) / (train count + α), α = 20, per payment format | Encodes the laundering risk of each payment method in a single float; ACH carries ~6× the global rate while Reinvestment and Wire are near zero — one column instead of seven, with no OHE sparsity |
| `Is_Self_Loop` | 1 if src\_account == dst\_account, else 0 | Reinvestment transactions (sender = receiver) have near-zero laundering rate and are structurally distinct from all other edges |


In [ ]:
edges = df_tr.copy()

# 1.1 Timestamp
edges['Timestamp'] = pd.to_datetime(edges['Timestamp'])

# Sort by timestamp here (needed for target encoding + later temporal features)
edges = edges.sort_values('Timestamp', kind='stable').reset_index(drop=True)

# Training cutoff index (60% of all edges) -- used below for leakage-free encoding
_t1_idx = int(len(edges) * 0.60)

# 1.2 Amount in USD + log1p
# Vectorised: build rate via unique (currency, date) pairs then left-merge
edges['_date_str'] = edges['Timestamp'].dt.date.astype(str)
_pairs = edges[['Payment Currency', '_date_str']].drop_duplicates().copy()
_pairs['_fx_rate'] = _pairs.apply(
    lambda r: fx_rate_lookup.get((r['Payment Currency'], r['_date_str']), 1.0),
    axis=1,
)
edges = edges.merge(_pairs, on=['Payment Currency', '_date_str'], how='left')
edges['_fx_rate'] = edges['_fx_rate'].fillna(1.0)   # USD rows: keep 1.0
edges['Amount_Log'] = np.log1p(edges['Amount Paid'] * edges['_fx_rate'])

# 1.3 Cross-currency flag
edges['Currency_Mismatch'] = (
    edges['Receiving Currency'] != edges['Payment Currency']
).astype(np.int8)

# 1.4 Cyclical time encoding
_hour = edges['Timestamp'].dt.hour
_dow  = edges['Timestamp'].dt.dayofweek
edges['Hour_Sin']      = np.sin(2 * np.pi * _hour / 24)
edges['Hour_Cos']      = np.cos(2 * np.pi * _hour / 24)
edges['DayOfWeek_Sin'] = np.sin(2 * np.pi * _dow  / 7)
edges['DayOfWeek_Cos'] = np.cos(2 * np.pi * _dow  / 7)
edges['Is_Weekend']    = (_dow >= 5).astype(np.int8)

# 1.5 Payment Format -- training-set target encoding (no OHE, no leakage)
# Formula: (train_sum + alpha * global_rate) / (train_count + alpha)
_ALPHA        = 20
_train_rows   = edges.index < _t1_idx
_global_rate  = edges.loc[_train_rows, 'Is Laundering'].mean()

_fmt_stats = (
    edges.loc[_train_rows]
    .groupby('Payment Format')['Is Laundering']
    .agg(['sum', 'count'])
)
_fmt_stats['PayFmt_TargetEnc'] = (
    (_fmt_stats['sum'] + _ALPHA * _global_rate) /
    (_fmt_stats['count'] + _ALPHA)
).astype(np.float32)
edges = edges.merge(
    _fmt_stats[['PayFmt_TargetEnc']],
    left_on='Payment Format', right_index=True, how='left',
)
edges['PayFmt_TargetEnc'] = (
    edges['PayFmt_TargetEnc'].fillna(_global_rate).astype(np.float32)
)

# 1.6 Self-loop
edges['Is_Self_Loop'] = (edges['Account'] == edges['Account.1']).astype(np.int8)

# 1.7 Rename
edges = edges.rename(columns={
    'Account':       'src_account',
    'Account.1':     'dst_account',
    'From Bank':     'src_bank',
    'To Bank':       'dst_bank',
    'Is Laundering': 'label',
})

BASE_EDGE_COLS = [
    'Amount_Log', 'Currency_Mismatch',
    'Hour_Sin', 'Hour_Cos', 'DayOfWeek_Sin', 'DayOfWeek_Cos',
    'Is_Weekend', 'PayFmt_TargetEnc', 'Is_Self_Loop',
]

print(f'Baseline edge features: {len(BASE_EDGE_COLS)}  '
      f'(OHE+Is_ACH removed; FX-corrected Amount_Log + PayFmt target enc added)')
print(f'  Global laundering rate (train): {_global_rate:.6f}')
print('  PayFmt encoding (train-set smoothed rates):')
for _fmt, _row in _fmt_stats.iterrows():
    print(f'    {_fmt:<22} -> {_row["PayFmt_TargetEnc"]:.6f}  (n={int(_row["count"]):,})')


---
## 2. Temporal account-state features  *(new)*

For every edge `(u→v, t)` we ask: **what did account u and account v look
like just before this transaction?**  We answer this for all four behavioural
directions (2 accounts × 2 roles), then add short-window burst and cycle features.

In [ ]:
# ── Critical: sort once by timestamp before any computation ──────────────────
edges   = edges.sort_values('Timestamp', kind='stable').reset_index(drop=True)
edges_t = edges.set_index('Timestamp')   # time-indexed view for rolling windows

print(f'Sorted by Timestamp ✓')
print(f'  Range  : {edges["Timestamp"].min()}  →  {edges["Timestamp"].max()}')
print(f'  Edges  : {len(edges):,}')

# ─────────────────────────────────────────────────────────────────────────────
# add_vertex_stats(df, df_t, account_col, counterpart_col, prefix)
# ─────────────────────────────────────────────────────────────────────────────
# Computes 11 rolling/lifetime statistics for `account_col` as the focal
# account.  Attaches columns to `df` in-place; returns list of new col names.
#
# Leakage guard:
#   .shift(1)          — excludes current row from its own expanding aggregate
#   .shift(freq='1ms') — excludes current row from its own rolling window
#
# Features produced (prefix='pfx', windows=['1D','3D'] → 11 cols):
#   pfx_count_life     : transaction count before this one  (0 on first ever)
#   pfx_vol_life       : total volume before this one
#   pfx_mean_life      : mean amount per transaction  (personal baseline)
#   pfx_std_life       : std of amounts  (low std = structuring)
#   pfx_unique_life    : distinct counterparties ever seen
#   pfx_count_1d       : count in last 1 day
#   pfx_vol_1d         : volume in last 1 day
#   pfx_diversity_ratio: unique_life / (count_life + 1)  in [0, 1]
#   pfx_spike_1d       : count_1d / (count_life/10 + 1)  — velocity burst

def _cum_nunique(x):
    """Vectorized cumulative unique count: result[i] = #distinct values in x[0..i-1].
    Uses np.unique+cumsum (numpy C code) instead of a per-row Python loop."""
    vals = x.values
    if len(vals) == 0:
        return pd.Series([], index=x.index, dtype=float)
    _, first_idx = np.unique(vals, return_index=True)
    is_first = np.zeros(len(vals), dtype=np.int8)
    is_first[first_idx] = 1
    result = np.empty(len(vals))
    result[0] = 0.0
    if len(vals) > 1:
        result[1:] = np.cumsum(is_first[:-1])
    return pd.Series(result, index=x.index)

def add_vertex_stats(df, df_t, account_col, counterpart_col, prefix, windows=None):
    if windows is None:
        windows = ['1D']
    new_cols = []
    grp   = df.groupby(account_col,   sort=False)
    grp_t = df_t.groupby(account_col, sort=False)

    col = f'{prefix}_count_life'
    df[col] = grp.cumcount()   # 0 = first ever txn for this account
    new_cols.append(col)

    col = f'{prefix}_vol_life'
    df[col] = grp['Amount Paid'].transform(
        lambda x: x.shift(1).expanding().sum().fillna(0))
    new_cols.append(col)

    col = f'{prefix}_mean_life'
    df[col] = grp['Amount Paid'].transform(
        lambda x: x.shift(1).expanding().mean().fillna(0))
    new_cols.append(col)

    col = f'{prefix}_std_life'
    df[col] = grp['Amount Paid'].transform(
        lambda x: x.shift(1).expanding().std().fillna(0))
    new_cols.append(col)

    col = f'{prefix}_unique_life'
    # O(n) cumulative unique — avoids expanding().apply() O(n²) memory blowup.
    df[col] = grp[counterpart_col].transform(_cum_nunique)
    new_cols.append(col)

    for w in windows:
        wl = w.lower()
        col = f'{prefix}_count_{wl}'
        # .to_numpy() inside lambda: pandas 2.x raises ValueError("cannot reindex on
        # an axis with duplicate labels") when shift(freq) changes the time index and
        # transform tries to reindex the result back to the group's duplicate timestamps.
        # Returning a plain numpy array bypasses the reindex step entirely.
        df[col] = (
            grp_t['Amount Paid']
            .transform(lambda x: x.shift(freq='1ms').rolling(w).count().fillna(0).to_numpy())
            .values
        )
        new_cols.append(col)

        col = f'{prefix}_vol_{wl}'
        df[col] = (
            grp_t['Amount Paid']
            .transform(lambda x: x.shift(freq='1ms').rolling(w).sum().fillna(0).to_numpy())
            .values
        )
        new_cols.append(col)

    col = f'{prefix}_diversity_ratio'
    df[col] = df[f'{prefix}_unique_life'] / (df[f'{prefix}_count_life'] + 1)
    new_cols.append(col)

    col = f'{prefix}_spike_1d'
    df[col] = df[f'{prefix}_count_1d'] / (df[f'{prefix}_count_life'] / 10 + 1)
    new_cols.append(col)

    return new_cols


# ─────────────────────────────────────────────────────────────────────────────
# align_cross_stats()
# ─────────────────────────────────────────────────────────────────────────────
# Sections 2.2 and 2.4 need stats of account X in a role it does NOT play in
# the current edge.  Strategy:
#   1. Compute stats on a scratch copy treating account_col_src as focal.
#   2. Align those stats to account_col_tgt in the main edges frame via
#      merge_asof on row index (= timestamp order), direction='backward'.
#      For each edge i where src_account=A, this finds the latest past row
#      where dst_account=A and reads its running incoming stats.

def align_cross_stats(edges, account_col_src, account_col_tgt,
                      counterpart_col, prefix_tmp, prefix_final):
    ec   = edges[['Timestamp', 'src_account', 'dst_account', 'Amount Paid']].copy()
    ec_t = ec.set_index('Timestamp')

    tmp_cols = add_vertex_stats(
        ec, ec_t,
        account_col    =account_col_src,
        counterpart_col=counterpart_col,
        prefix         =prefix_tmp,
    )

    lkp = ec[[account_col_src] + tmp_cols].copy()
    rename_map = {account_col_src: 'focal_account'}
    rename_map.update(
        {c: c.replace(prefix_tmp + '_', prefix_final + '_') for c in tmp_cols}
    )
    lkp = lkp.rename(columns=rename_map)
    final_cols = [rename_map[c] for c in tmp_cols]

    lkp['_ridx']   = np.arange(len(lkp))
    edges['_ridx'] = np.arange(len(edges))

    lkp_sorted = (
        lkp.rename(columns={'focal_account': account_col_tgt})
           .sort_values('_ridx')
    )
    merged = pd.merge_asof(
        edges[[account_col_tgt, '_ridx']].sort_values('_ridx'),
        lkp_sorted,
        on='_ridx', by=account_col_tgt, direction='backward',
    )
    for col in final_cols:
        edges[col] = merged[col].fillna(0).values

    edges.drop(columns=['_ridx'], inplace=True)
    return final_cols


print('Helper functions defined ✓')

### 2.1  Sender — outgoing history

What has account `u` been doing as a **sender** before time `t`?

| Feature | Intuition |
|---|---|
| `src_out_count_life` | How active is u as a sender overall? 0 = brand new account |
| `src_out_vol_life` | Total funds u has moved out historically |
| `src_out_mean_life` | u's personal average send amount — deviation from this is suspicious |
| `src_out_std_life` | Low std = suspiciously uniform amounts (structuring: deliberately keeping amounts below reporting thresholds) |
| `src_out_unique_life` | How many distinct accounts has u ever sent to? High = fan-out behaviour |
| `src_out_count_1d` | How many times has u sent in the last 24 h? Burst = suspicious |
| `src_out_vol_1d` | How much has u sent in the last 24 h? |
| `src_out_count_3d` | Medium-term sending activity |
| `src_out_vol_3d` | Medium-term sending volume |
| `src_out_diversity_ratio` | unique_life / (count_life+1) → 1.0 = every send to a new account (fan-out); 0 = always the same counterparty |
| `src_out_spike_1d` | count_1d / (count_life/10 + 1) → today unusually busy vs personal norm |

In [ ]:
print('2.1  Sender outgoing history ...')
with Timer('2.1 src_out'):
    SRC_OUT_COLS = add_vertex_stats(
        edges, edges_t,
        account_col    ='src_account',
        counterpart_col='dst_account',
        prefix         ='src_out',
    )
print(f'     → {len(SRC_OUT_COLS)} features:  {SRC_OUT_COLS}')

### 2.2  Sender — incoming history

What has account `u` been **receiving** before time `t`?

**Why this matters:** the sender may look normal from its outgoing side, but
its incoming history reveals where its funds came from.  If u was itself a
fan-in target just before forwarding the money, the upstream laundering chain
is visible only through these features.  Combining `src_in_vol_1d` with
`src_out_vol_1d` directly detects a same-day receive-then-forward (pass-through)
pattern.

| Feature | Intuition |
|---|---|
| `src_in_count_life` | Has u been receiving frequently? |
| `src_in_vol_life` | Total funds that have flowed into u |
| `src_in_mean_life` | Average receipt amount |
| `src_in_std_life` | Uniformity of incoming amounts |
| `src_in_unique_life` | How many distinct senders has u received from? (upstream fan-in) |
| `src_in_count_1d` | Did u receive a burst of funds today before sending? |
| `src_in_vol_1d` | How much did u receive today? Pair with `src_out_vol_1d` for pass-through detection |
| `src_in_count_3d` | Medium-term incoming activity |
| `src_in_vol_3d` | Medium-term incoming volume |
| `src_in_diversity_ratio` | Unique sources / total receives → fan-in signal for the sender |
| `src_in_spike_1d` | Incoming burst vs personal receiving norm |

In [ ]:
# Strategy: u's incoming history = stats of u when it appeared as dst_account.
# Computed on a scratch copy (account_col_src='dst_account'), then aligned
# back to src_account via merge_asof on row index.

print('2.2  Sender incoming history ...')
with Timer('2.2 src_in'):
    SRC_IN_COLS = align_cross_stats(
        edges,
        account_col_src='dst_account',   # focal account is the receiver in past edges
        account_col_tgt='src_account',   # attach result to sender column in main frame
        counterpart_col='src_account',
        prefix_tmp     ='_srci',
        prefix_final   ='src_in',
    )
print(f'     → {len(SRC_IN_COLS)} features:  {SRC_IN_COLS}')

### 2.3  Receiver — incoming history

What has account `v` been **receiving** before time `t`?

**Why this matters:** Fan-In laundering manifests here directly.  A mule
accumulates many distinct sources sending it money.  `dst_in_unique_life`
and `dst_in_spike_1d` are the two most discriminative single features for
the Fan-In typology in this dataset.

| Feature | Intuition |
|---|---|
| `dst_in_count_life` | How often has v been receiving overall? |
| `dst_in_vol_life` | Total funds received by v historically |
| `dst_in_mean_life` | Average receipt amount (v's receiving baseline) |
| `dst_in_std_life` | Uniformity of receipts |
| `dst_in_unique_life` | How many distinct senders has v ever received from? — primary Fan-In indicator |
| `dst_in_count_1d` | Incoming burst in last 24 h — core Fan-In signal |
| `dst_in_vol_1d` | Large recent incoming volume |
| `dst_in_count_3d` | Medium-term inflow activity |
| `dst_in_vol_3d` | Medium-term inflow volume |
| `dst_in_diversity_ratio` | unique_life / (count_life+1) → 1 = classic fan-in mule |
| `dst_in_spike_1d` | Today unusually busy vs v's receiving norm |

In [ ]:
print('2.3  Receiver incoming history ...')
with Timer('2.3 dst_in'):
    DST_IN_COLS = add_vertex_stats(
        edges, edges_t,
        account_col    ='dst_account',
        counterpart_col='src_account',
        prefix         ='dst_in',
    )
print(f'     → {len(DST_IN_COLS)} features:  {DST_IN_COLS}')

### 2.4  Receiver — outgoing history

What has account `v` been doing as a **sender** before time `t`?

**Why this matters:** a receiver that quickly re-sends whatever it receives
is a pass-through mule.  Without this block the model sees the gather but not
the scatter.  Combining `dst_in_vol_1d` (received recently) with `dst_out_vol_1d`
(sent recently) is the strongest pass-through / Gather-Scatter signal.

| Feature | Intuition |
|---|---|
| `dst_out_count_life` | How active is v as a sender overall? |
| `dst_out_vol_life` | Total funds v has pushed out historically |
| `dst_out_mean_life` | Average outgoing amount (v's sending baseline) |
| `dst_out_std_life` | Uniformity of outgoing amounts |
| `dst_out_unique_life` | How many distinct destinations has v ever sent to? (fan-out history) |
| `dst_out_count_1d` | Sending burst in last 24 h |
| `dst_out_vol_1d` | Recent outgoing volume — pair with `dst_in_vol_1d` for pass-through |
| `dst_out_count_3d` | Medium-term sending activity |
| `dst_out_vol_3d` | Medium-term sending volume |
| `dst_out_diversity_ratio` | Fan-out ratio for the receiver |
| `dst_out_spike_1d` | Sending burst vs v's norm |

In [ ]:
# Mirror of Section 2.2: v's outgoing history = stats of v when it appeared as
# src_account in past edges.  Computed on a scratch copy then aligned back.

print('2.4  Receiver outgoing history ...')
with Timer('2.4 dst_out'):
    DST_OUT_COLS = align_cross_stats(
        edges,
        account_col_src='src_account',   # focal account is the sender in past edges
        account_col_tgt='dst_account',   # attach result to receiver column in main frame
        counterpart_col='dst_account',
        prefix_tmp     ='_dsto',
        prefix_final   ='dst_out',
    )
print(f'     → {len(DST_OUT_COLS)} features:  {DST_OUT_COLS}')

### 2.5  Scatter-gather features  (6-hour window, both sides)

Replicates the GFP scatter-gather feature group from Altman et al. Appendix D
using a **6-hour window** for both the sender (scatter) and receiver (gather).

**Why 6 hours?**  Layering sequences are rapid.  Funds scattered to many
intermediaries typically reconsolidate within hours, not days.  A 6-hour
window captures bursts without accumulating background noise from unrelated activity.

| Feature | Intuition |
|---|---|
| `src_fanout_uniq_6h` | Distinct destinations u sent to in last 6 h — rapid scatter |
| `src_vol_6h` | Total volume u sent in last 6 h — scatter burst size |
| `src_count_6h` | Transaction count u sent in last 6 h — high-frequency burst |
| `dst_fanin_uniq_6h` | Distinct sources that sent to v in last 6 h — rapid gather |
| `dst_vol_6h` | Total volume v received in last 6 h — gather burst size |
| `dst_count_6h` | Transaction count v received in last 6 h |
| `sg_combined_6h` | √(src_fanout_uniq_6h × dst_fanin_uniq_6h) — high only when BOTH sides burst simultaneously; the clearest single scatter-gather indicator |
| `dst_resend_ratio_1d` | dst_out_vol_1d / (dst_in_vol_1d + 1) — fraction of what v received recently that it re-sent; >1 = pass-through mule |

In [ ]:
print('2.5  Scatter-gather features (6h window, both sides) ...')
with Timer('2.5 scatter-gather'):

    # Encode string account IDs as integer category codes for rolling().apply().
    # rolling().apply() requires numeric dtype in pandas >= 2.0; set(int_codes)
    # counts the same number of unique values as set(string_ids).
    _src_codes = edges['src_account'].astype('category').cat.codes.astype(float).values
    _dst_codes = edges['dst_account'].astype('category').cat.codes.astype(float).values
    edges_t_enc = edges_t.copy()
    edges_t_enc['_src_code'] = _src_codes
    edges_t_enc['_dst_code'] = _dst_codes

    # ── Sender (scatter) side ─────────────────────────────────────────────────
    edges['src_fanout_uniq_6h'] = (
        edges_t_enc.groupby('src_account')['_dst_code']
        .transform(
            lambda x: x.shift(freq='1ms').rolling('6H')
                       .apply(lambda s: float(len(set(s[~np.isnan(s)]))), raw=True)
                       .fillna(0).to_numpy()
        ).values
    )
    edges['src_vol_6h'] = (
        edges_t_enc.groupby('src_account')['Amount Paid']
        .transform(lambda x: x.shift(freq='1ms').rolling('6H').sum().fillna(0).to_numpy())
        .values
    )
    edges['src_count_6h'] = (
        edges_t_enc.groupby('src_account')['Amount Paid']
        .transform(lambda x: x.shift(freq='1ms').rolling('6H').count().fillna(0).to_numpy())
        .values
    )

    # ── Receiver (gather) side ────────────────────────────────────────────────
    edges['dst_fanin_uniq_6h'] = (
        edges_t_enc.groupby('dst_account')['_src_code']
        .transform(
            lambda x: x.shift(freq='1ms').rolling('6H')
                       .apply(lambda s: float(len(set(s[~np.isnan(s)]))), raw=True)
                       .fillna(0).to_numpy()
        ).values
    )
    edges['dst_vol_6h'] = (
        edges_t_enc.groupby('dst_account')['Amount Paid']
        .transform(lambda x: x.shift(freq='1ms').rolling('6H').sum().fillna(0).to_numpy())
        .values
    )
    edges['dst_count_6h'] = (
        edges_t_enc.groupby('dst_account')['Amount Paid']
        .transform(lambda x: x.shift(freq='1ms').rolling('6H').count().fillna(0).to_numpy())
        .values
    )

    # ── Combined signals ──────────────────────────────────────────────────────
    # Geometric mean: high only when BOTH scatter and gather burst simultaneously
    edges['sg_combined_6h'] = np.sqrt(
        edges['src_fanout_uniq_6h'] * edges['dst_fanin_uniq_6h']
    )

    # Pass-through ratio: uses dst_out_vol_1d (Sec 2.4) and dst_in_vol_1d (Sec 2.3)
    # NOTE: Sections 2.3 and 2.4 must run before this cell
    edges['dst_resend_ratio_1d'] = (
        edges['dst_out_vol_1d'] / (edges['dst_in_vol_1d'] + 1)
    )

SG_COLS = [
    'src_fanout_uniq_6h', 'src_vol_6h',    'src_count_6h',
    'dst_fanin_uniq_6h',  'dst_vol_6h',    'dst_count_6h',
    'sg_combined_6h',     'dst_resend_ratio_1d',
]
print(f'     → {len(SG_COLS)} features:  {SG_COLS}')

### 2.6  Cycle / structural features  (both accounts, 1-day sliding window)

Detects whether the current transaction closes a directed cycle in the
1-day historical graph, and tracks cycle participation for **both** accounts.

**Symmetry rationale:** an account repeatedly seen in cycles is suspicious
regardless of whether it is sending or receiving right now.  Previous version
only tracked `src_cycle_count`; this version adds `dst_cycle_count` so the
receiver's cycle history is also available to the model.

**Algorithm (causal, zero leakage):**
1. Maintain a sliding directed graph `G` of edges within the last 1 day.
2. For edge `(u→v, t)`: depth-limited BFS from `v` in `G` searching for `u`.
3. Record `src_cycle_count` and `dst_cycle_count` **before** adding this edge.
4. If a cycle found: increment both accounts' cycle counts.
5. **Then** add `(u→v)` to `G` and prune edges older than 1 day.

| Feature | Intuition |
|---|---|
| `closes_cycle` | Binary: does this transaction complete a directed money loop in the last day? |
| `min_cycle_len` | Length of shortest cycle closed; len=2 (A→B→A) is immediate return, very suspicious |
| `src_cycle_count` | How many cycles has the **sender** already participated in today? High = chronic cycle actor |
| `dst_cycle_count` | How many cycles has the **receiver** already participated in today? Symmetric signal for the receiving side |

> **Runtime note:** `MAX_DEPTH=4` takes ~15–30 min for 6.9M edges.  
> Reduce to `MAX_DEPTH=3` to roughly halve the runtime with minor coverage loss.

In [ ]:
MAX_DEPTH   = 4          # max cycle length to detect
TIME_WINDOW = pd.Timedelta('1D')

def bfs_has_path(adj, source, target, max_depth):
    """Depth-limited BFS from source in adjacency dict adj.
    Returns (found: bool, hops: int)."""
    if source == target: return True, 0
    visited = {source}
    queue   = deque([(source, 0)])
    while queue:
        node, depth = queue.popleft()
        if depth >= max_depth: continue
        for nb in adj.get(node, {}):
            if nb == target: return True, depth + 1
            if nb not in visited:
                visited.add(nb)
                queue.append((nb, depth + 1))
    return False, 0


print(f'2.6  Cycle features (MAX_DEPTH={MAX_DEPTH}, window=1 day) ...')
print(f'     Expected runtime: ~15-30 min for 6.9M edges.')
print(f'     Tip: set MAX_DEPTH=3 above to halve runtime.')

# Pre-allocate output arrays (much faster than Python list appends at this scale)
n = len(edges)
closes_arr  = np.zeros(n, dtype=np.int8)
cyc_len_arr = np.zeros(n, dtype=np.int16)
src_cyc_arr = np.zeros(n, dtype=np.int32)
dst_cyc_arr = np.zeros(n, dtype=np.int32)

src_arr = edges['src_account'].values
dst_arr = edges['dst_account'].values
ts_arr  = edges['Timestamp'].values
win_ns  = int(TIME_WINDOW.total_seconds() * 1e9)

edge_win: deque = deque()   # (ts_ns, src, dst)
adj: dict       = {}        # adj[u] = {v: count} -- O(1) add/remove
node_cyc: dict  = {}        # node -> cycle count in current window

for i in tqdm(range(n), miniters=100_000, desc='Cycle detection'):
    u = src_arr[i]
    v = dst_arr[i]
    t = int(ts_arr[i])

    # Prune edges outside the 1-day sliding window
    cutoff = t - win_ns
    while edge_win and edge_win[0][0] < cutoff:
        _, ou, ov = edge_win.popleft()
        if ou in adj:
            cnt = adj[ou].get(ov, 0) - 1
            if cnt > 0:
                adj[ou][ov] = cnt
            else:
                adj[ou].pop(ov, None)

    # Snapshot cycle participation counts BEFORE checking this edge
    # (both accounts get their current count regardless of whether a new
    # cycle is found — this ensures the recorded count is the history up to t)
    src_cyc_arr[i] = node_cyc.get(u, 0)
    dst_cyc_arr[i] = node_cyc.get(v, 0)

    # BFS: does path v ->...-> u exist in past graph? = does (u->v) close a cycle?
    found, hops    = bfs_has_path(adj, v, u, MAX_DEPTH)
    closes_arr[i]  = int(found)
    cyc_len_arr[i] = hops + 1 if found else 0

    # Increment cycle counts for BOTH endpoints if cycle detected
    if found:
        node_cyc[u] = node_cyc.get(u, 0) + 1
        node_cyc[v] = node_cyc.get(v, 0) + 1

    # Add (u->v) to graph AFTER computing its features (causal guarantee)
    nbrs = adj.setdefault(u, {}); nbrs[v] = nbrs.get(v, 0) + 1
    edge_win.append((t, u, v))

edges['closes_cycle']    = closes_arr
edges['min_cycle_len']   = cyc_len_arr
edges['src_cycle_count'] = src_cyc_arr
edges['dst_cycle_count'] = dst_cyc_arr   # symmetric: receiver cycle history

CYCLE_COLS = ['closes_cycle', 'min_cycle_len', 'src_cycle_count', 'dst_cycle_count']

n_cyc = int(closes_arr.sum())
print(f'\n     Edges closing a cycle : {n_cyc:,} ({100*n_cyc/n:.3f}%)')
print(f'     → {len(CYCLE_COLS)} cycle features added')

### 2.7  Fill NaN & final feature summary

In [ ]:
ALL_TEMPORAL_COLS = (
    SRC_OUT_COLS + SRC_IN_COLS +
    DST_IN_COLS  + DST_OUT_COLS +
    SG_COLS      + CYCLE_COLS
)

# Safety-net fill: first-transaction rows have 0 from the helpers;
# this catches any edge case from the merge_asof alignment.
nan_before = edges[ALL_TEMPORAL_COLS].isna().sum().sum()
edges[ALL_TEMPORAL_COLS] = edges[ALL_TEMPORAL_COLS].fillna(0).astype(np.float32)
print(f'NaN values filled: {nan_before:,} → 0')
print()
print('── Feature counts per group ──────────────────────────────────────')
print(f'  2.1  src_out  (sender outgoing)   : {len(SRC_OUT_COLS):>3}')
print(f'  2.2  src_in   (sender incoming)   : {len(SRC_IN_COLS):>3}')
print(f'  2.3  dst_in   (receiver incoming) : {len(DST_IN_COLS):>3}')
print(f'  2.4  dst_out  (receiver outgoing) : {len(DST_OUT_COLS):>3}')
print(f'  2.5  scatter-gather (6h)          : {len(SG_COLS):>3}')
print(f'  2.6  cycle / structural           : {len(CYCLE_COLS):>3}')
print(f'  ──────────────────────────────────────')
print(f'  New temporal total                : {len(ALL_TEMPORAL_COLS):>3}')
print(f'  Baseline                          : {len(BASE_EDGE_COLS):>3}')
print(f'  GRAND TOTAL edge_attr dim         : {len(BASE_EDGE_COLS)+len(ALL_TEMPORAL_COLS):>3}')

---
## 3. Node features  *(v2: training-set target encodings)*

Two account-level features derived from **training edges only** — no information from the
validation or test splits leaks into the node representations.

Both use **Bayesian smoothing** (α = 20) to produce stable estimates even for banks or entity
types that appear rarely in the training set.  The smoothed estimate shrinks towards the global
laundering rate as the sample size decreases:

> encoded value = (laund\_count + α × global\_rate) / (edge\_count + α)

**Changes from v1:** replaces `Bank_ID_Norm` (min-max scaled integer ID — semantically
meaningless) and `Entity_Type` OHE (5 binary columns) with two compact, information-rich
target-encoded scalars.

| Feature | How it is computed | AML intuition |
|---|---|---|
| `Bank_TargetEnc` | Smoothed laundering rate across all training edges where the account's bank appears as either `src_bank` or `dst_bank`.  Both sides pooled before aggregation so that a bank's risk is estimated from all its activity, not just outgoing or incoming flows alone. | Banks differ substantially in AML controls, jurisdictions, and customer profiles.  A single float captures institutional risk far better than a normalised integer ID, and better than a one-hot encoding that would require a column per bank. |
| `EntityType_TargetEnc` | Entity type (Corporation / Individual / Partnership / Sole Proprietor / Other) is extracted from the `Entity Name` field in the accounts table.  For each training edge the entity types of both src and dst accounts are looked up and their labels pooled; the smoothed group rate is then assigned to every account of that type. | Entity type correlates with typical transaction patterns and regulatory exposure.  Encoding the observed laundering rate rather than a binary indicator gives the model a continuous risk signal aligned with the target variable. |


In [ ]:
nodes = df_ac.copy()

def _extract_entity_type(name):
    for t in ('Corporation', 'Individual', 'Partnership', 'Sole'):
        if t.lower() in str(name).lower():
            return t
    return 'Other'

nodes['Entity_Type'] = nodes['Entity Name'].apply(_extract_entity_type)

# Training edges for target encoding (uses _t1_idx from original-features cell)
_ALPHA_N      = 20
_train_edges_n = edges[edges.index < _t1_idx][
    ['src_account', 'dst_account', 'label', 'src_bank', 'dst_bank']
].copy()
_global_rate_n = _train_edges_n['label'].mean()

# ── Bank target encoding ──────────────────────────────────────────────────────
# Pool both src_bank and dst_bank observations from training edges
_bk_src   = _train_edges_n[['src_bank', 'label']].rename(columns={'src_bank': 'bank_id'})
_bk_dst   = _train_edges_n[['dst_bank', 'label']].rename(columns={'dst_bank': 'bank_id'})
_bk_all   = pd.concat([_bk_src, _bk_dst], ignore_index=True)
_bk_stats = _bk_all.groupby('bank_id')['label'].agg(['sum', 'count'])
_bk_stats['Bank_TargetEnc'] = (
    (_bk_stats['sum'] + _ALPHA_N * _global_rate_n) /
    (_bk_stats['count'] + _ALPHA_N)
).astype(np.float32)
nodes = nodes.merge(
    _bk_stats[['Bank_TargetEnc']],
    left_on='Bank ID', right_index=True, how='left',
)
nodes['Bank_TargetEnc'] = nodes['Bank_TargetEnc'].fillna(_global_rate_n).astype(np.float32)

# ── Entity type target encoding ───────────────────────────────────────────────
# Map each account in training edges to its entity type, then aggregate labels
# Drop duplicates on Account Number to ensure unique index for .map()
_acc_type_map = (
    nodes.drop_duplicates(subset='Account Number')
         .set_index('Account Number')['Entity_Type']
)
_et_src  = _train_edges_n[['src_account', 'label']].copy()
_et_src['Entity_Type'] = _et_src['src_account'].map(_acc_type_map)
_et_dst  = _train_edges_n[['dst_account', 'label']].copy()
_et_dst['Entity_Type'] = _et_dst['dst_account'].map(_acc_type_map)
_et_all  = pd.concat(
    [_et_src[['Entity_Type', 'label']], _et_dst[['Entity_Type', 'label']]],
    ignore_index=True,
).dropna(subset=['Entity_Type'])
_et_stats = _et_all.groupby('Entity_Type')['label'].agg(['sum', 'count'])
_et_stats['EntityType_TargetEnc'] = (
    (_et_stats['sum'] + _ALPHA_N * _global_rate_n) /
    (_et_stats['count'] + _ALPHA_N)
).astype(np.float32)
nodes = nodes.merge(
    _et_stats[['EntityType_TargetEnc']],
    left_on='Entity_Type', right_index=True, how='left',
)
nodes['EntityType_TargetEnc'] = (
    nodes['EntityType_TargetEnc'].fillna(_global_rate_n).astype(np.float32)
)

NODE_FEAT_COLS_V2 = ['Bank_TargetEnc', 'EntityType_TargetEnc']
node_features_v2  = nodes[['Account Number'] + NODE_FEAT_COLS_V2].copy()
node_features_v2  = node_features_v2.rename(columns={'Account Number': 'account_id'})

print(f'Node feature matrix v2: {node_features_v2.shape}')
print(f'  Bank encoding  -- unique values: {nodes["Bank_TargetEnc"].nunique():,}')
print('  EntityType encoding (train-set smoothed rates):')
for _et, _row in _et_stats.iterrows():
    print(f'    {_et:<15} -> {_row["EntityType_TargetEnc"]:.6f}  (n={int(_row["count"]):,})')


In [ ]:
# Join new baseline (9 cols) with pre-computed temporal features from v1 CSV.
# Saves edge_features_enhanced_v2.csv and node_features_enhanced_v2.csv.
# Does NOT overwrite the existing v1 files.

print('Loading edge_features_enhanced.csv ...')
_ef_v1 = pd.read_csv('Data/edge_features_enhanced.csv', low_memory=False)
print(f'  Shape: {_ef_v1.shape}')

# Identify temporal columns: all columns except meta + old baseline
_META_V1    = {'src_account', 'dst_account', 'label', 'Timestamp'}
_OLD_BASE_V1 = (
    {'Amount_Log', 'Currency_Mismatch', 'Hour_Sin', 'Hour_Cos',
     'DayOfWeek_Sin', 'DayOfWeek_Cos', 'Is_Weekend', 'Is_ACH', 'Is_Self_Loop'}
    | {c for c in _ef_v1.columns if c.startswith('PayFmt_')}
)
_TEMPORAL_COLS = [c for c in _ef_v1.columns if c not in _META_V1 and c not in _OLD_BASE_V1]
print(f'  Temporal columns: {len(_TEMPORAL_COLS)}')

# Verify row alignment: both frames sorted by Timestamp from same source data
assert len(_ef_v1) == len(edges), (
    f'Row count mismatch: CSV={len(_ef_v1):,}  edges={len(edges):,}'
)
_align_ok = (
    (_ef_v1['src_account'].iloc[:10].values == edges['src_account'].iloc[:10].values).all()
    and
    (_ef_v1['src_account'].iloc[-10:].values == edges['src_account'].iloc[-10:].values).all()
)
if not _align_ok:
    raise ValueError('Row alignment check FAILED -- re-check Timestamp sort order!')
print('  Row alignment verified ✓')

# Build v2 edge feature frame: new baseline + temporal cols
_ef_v2 = pd.concat([
    edges[['src_account', 'dst_account', 'label', 'Timestamp'] + BASE_EDGE_COLS]
        .reset_index(drop=True),
    _ef_v1[_TEMPORAL_COLS].reset_index(drop=True),
], axis=1)

print(f'\nNew edge features: {_ef_v2.shape}')
print(f'  Baseline: {len(BASE_EDGE_COLS)} cols  |  Temporal (reused): {len(_TEMPORAL_COLS)} cols')

_ef_v2.to_csv('Data/edge_features_enhanced_v2.csv', index=False)
node_features_v2.to_csv('Data/node_features_enhanced_v2.csv', index=False)
print('Saved  Data/edge_features_enhanced_v2.csv')
print('Saved  Data/node_features_enhanced_v2.csv')


In [ ]:
# Build PyG graph snapshots for v2 features.
# Saves train/val/test_graph_enhanced_v2.pt + account_to_idx_enhanced_v2.pkl.

print('Loading v2 feature files ...')
_ef2 = pd.read_csv('Data/edge_features_enhanced_v2.csv', low_memory=False)
_nf2 = pd.read_csv('Data/node_features_enhanced_v2.csv')
_ef2['Timestamp'] = pd.to_datetime(_ef2['Timestamp'], format='mixed')
_ef2 = _ef2.sort_values('Timestamp', kind='stable').reset_index(drop=True)

_EDGE_FEAT_V2 = [c for c in _ef2.columns
                 if c not in {'src_account', 'dst_account', 'label', 'Timestamp'}]
_NODE_FEAT_V2 = [c for c in _nf2.columns if c != 'account_id']
print(f'  Edge features: {len(_EDGE_FEAT_V2)} | Node features: {len(_NODE_FEAT_V2)}')

# 60/20/20 temporal split
_n2, _t1_2, _t2_2 = len(_ef2), int(len(_ef2)*0.60), int(len(_ef2)*0.80)
_tr2 = _ef2.index < _t1_2
_va2 = (_ef2.index >= _t1_2) & (_ef2.index < _t2_2)

# Node index map (union of accounts table + all edge endpoints)
_all_accs2 = pd.concat([
    _nf2['account_id'], _ef2['src_account'], _ef2['dst_account']
]).unique()
_a2i = {acc: idx for idx, acc in enumerate(_all_accs2)}
_N2  = len(_a2i)
print(f'  Total unique accounts: {_N2:,}')

# Node feature matrix
_nf2_dd = (
    _nf2.drop_duplicates(subset='account_id', keep='first')
    if _nf2['account_id'].duplicated().sum() > 0 else _nf2
)
_Xn2 = torch.tensor(
    _nf2_dd.set_index('account_id')
           .reindex(pd.Series(_a2i).index)[_NODE_FEAT_V2]
           .values.astype(np.float32),
    dtype=torch.float,
)
print(f'  Node feature matrix: {tuple(_Xn2.shape)}')

with open('Data/account_to_idx_enhanced_v2.pkl', 'wb') as _fv2:
    pickle.dump(_a2i, _fv2)

def _build_v2(edge_sub, eval_mask):
    _src = edge_sub['src_account'].map(_a2i).values
    _dst = edge_sub['dst_account'].map(_a2i).values
    _ei  = torch.tensor(np.stack([_src, _dst], axis=0), dtype=torch.long)
    _ea  = torch.tensor(edge_sub[_EDGE_FEAT_V2].values.astype(np.float32), dtype=torch.float)
    _et  = torch.tensor(
        edge_sub['Timestamp'].astype(np.int64).values // 10**9, dtype=torch.long
    )
    _lbl = np.full(len(edge_sub), -1, dtype=np.int64)
    _lbl[eval_mask] = edge_sub.loc[eval_mask, 'label'].values.astype(np.int64)
    return Data(
        x=_Xn2, edge_index=_ei, edge_attr=_ea, edge_time=_et,
        y=torch.tensor(_lbl, dtype=torch.long),
        eval_mask=torch.tensor(eval_mask, dtype=torch.bool),
        num_nodes=_N2,
    )

print('\nBuilding v2 graph snapshots ...')
with Timer('train v2'):
    _tr_e2 = _ef2[_tr2].reset_index(drop=True)
    train_graph_v2 = _build_v2(_tr_e2, np.ones(len(_tr_e2), dtype=bool))

with Timer('val v2'):
    _va_e2  = _ef2[_tr2 | _va2].reset_index(drop=True)
    _va_ev2 = np.zeros(len(_va_e2), dtype=bool); _va_ev2[_t1_2:] = True
    val_graph_v2 = _build_v2(_va_e2, _va_ev2)

with Timer('test v2'):
    _te_ev2 = np.zeros(_n2, dtype=bool); _te_ev2[_t2_2:] = True
    test_graph_v2 = _build_v2(_ef2.reset_index(drop=True), _te_ev2)

torch.save(train_graph_v2, 'Data/train_graph_enhanced_v2.pt')
torch.save(val_graph_v2,   'Data/val_graph_enhanced_v2.pt')
torch.save(test_graph_v2,  'Data/test_graph_enhanced_v2.pt')

def _sum_v2(name, g):
    _ne = g.eval_mask.sum().item()
    _nl = (g.y[g.eval_mask] == 1).sum().item()
    print(f'  {name:<22} | nodes={g.num_nodes:>7,} | edges={g.edge_index.shape[1]:>9,} '
          f'| eval={_ne:>9,} | laund={_nl:>5,} ({100*_nl/_ne:.4f}%)')

print('\n── v2 Graph snapshots ─────────────────────────')
_sum_v2('train_graph_v2', train_graph_v2)
_sum_v2('val_graph_v2',   val_graph_v2)
_sum_v2('test_graph_v2',  test_graph_v2)
print(f'\nEdge feature dim : {train_graph_v2.edge_attr.shape[1]}')
print(f'Node feature dim : {train_graph_v2.x.shape[1]}')
print('\nSaved:')
print('  Data/train_graph_enhanced_v2.pt')
print('  Data/val_graph_enhanced_v2.pt')
print('  Data/test_graph_enhanced_v2.pt')
print('  Data/account_to_idx_enhanced_v2.pkl')
print(f'\nACTION: set EDGE_DIM = {train_graph_v2.edge_attr.shape[1]} in your v2 model notebook')


---
## 4. Assemble & save  *(v2)*

Combines the v2 baseline features (Section 1) with the temporal features
(Sections 2.1–2.6) and saves the final CSVs.  Outputs overwrite **v2 files only**
— the original v1 files (`*_enhanced.csv`, `*_enhanced.pt`) are not touched.


In [ ]:
EDGE_FEAT_COLS = BASE_EDGE_COLS + ALL_TEMPORAL_COLS

edge_features = edges[
    ['src_account', 'dst_account', 'label', 'Timestamp'] + EDGE_FEAT_COLS
].copy()
edge_features[EDGE_FEAT_COLS] = edge_features[EDGE_FEAT_COLS].astype(np.float32)

edge_features.to_csv('Data/edge_features_enhanced_v2.csv', index=False)
node_features_v2.to_csv('Data/node_features_enhanced_v2.csv', index=False)

print(f'Saved  edge_features_enhanced_v2.csv : {edge_features.shape}')
print(f'Saved  node_features_enhanced_v2.csv : {node_features_v2.shape}')
vc = edge_features['label'].value_counts()
print(f'Laundering rate: {vc[1]/vc.sum()*100:.4f}%  ({vc[1]:,} / {vc.sum():,})')


---
## 5. Graph construction  *(v2)*

Builds three PyTorch Geometric graph snapshots (train / val / test) from the
v2 feature files and saves them as `*_enhanced_v2.pt`.

> After running, update `EDGE_DIM` in your model notebook to the grand
> total printed at the end of Section 2.7 (currently **57**).


In [ ]:
edge_df = edge_features.copy()
edge_df['Timestamp'] = pd.to_datetime(edge_df['Timestamp'])

# Sort by timestamp (stable) — mirrors Data_prepration.ipynb exactly
edge_df = edge_df.sort_values('Timestamp', kind='stable').reset_index(drop=True)

n_edges = len(edge_df)
t1_idx  = int(n_edges * 0.60)
t2_idx  = int(n_edges * 0.80)

t1 = edge_df.loc[t1_idx - 1, 'Timestamp']
t2 = edge_df.loc[t2_idx - 1, 'Timestamp']

train_mask = edge_df.index < t1_idx
val_mask   = (edge_df.index >= t1_idx) & (edge_df.index < t2_idx)
test_mask  = edge_df.index >= t2_idx

print('60/20/20 temporal split:')
print(f'  t1 (train end) : {t1}  ->  {t1_idx:,} training edges')
print(f'  t2 (val end)   : {t2}  ->  {t2_idx - t1_idx:,} validation edges')
print(f'  t_max          : {edge_df["Timestamp"].max()}  ->  {n_edges - t2_idx:,} test edges')
print(f'
  Laundering in train : {edge_df.loc[train_mask, "label"].sum():,} | Rate: {edge_df.loc[train_mask, "label"].mean()*100:.4f}%')
print(f'  Laundering in val   : {edge_df.loc[val_mask,   "label"].sum():,}   | Rate: {edge_df.loc[val_mask,   "label"].mean()*100:.4f}%')
print(f'  Laundering in test  : {edge_df.loc[test_mask,  "label"].sum():,}   | Rate: {edge_df.loc[test_mask,  "label"].mean()*100:.4f}%')


In [ ]:
# Union of all account IDs in the accounts table OR in edges — mirrors Data_prepration.ipynb
all_accounts = pd.concat([
    node_features_v2['account_id'],
    edge_df['src_account'],
    edge_df['dst_account'],
]).unique()

account_to_idx = {acc: idx for idx, acc in enumerate(all_accounts)}
N_nodes = len(account_to_idx)
print(f'Total unique accounts (nodes): {N_nodes:,}')

NODE_FEAT_COLS = [c for c in node_features_v2.columns if c != 'account_id']

# Handle duplicate account_ids (same approach as Data_prepration.ipynb)
n_dupes = node_features_v2['account_id'].duplicated().sum()
if n_dupes > 0:
    print(f'  Warning: {n_dupes:,} duplicate account_id rows -- keeping first')
    node_feat_deduped = node_features_v2.drop_duplicates(subset='account_id', keep='first')
else:
    node_feat_deduped = node_features_v2

idx_series = pd.Series(account_to_idx)

node_df_indexed = (
    node_feat_deduped
    .set_index('account_id')
    .reindex(idx_series.index)
    [NODE_FEAT_COLS]
    .values
    .astype(np.float32)
)

X_node = torch.tensor(node_df_indexed, dtype=torch.float)
print(f'Node feature matrix : {tuple(X_node.shape)}')


In [ ]:
# Save account_to_idx for v2 graphs
with open("Data/account_to_idx_enhanced_v2.pkl", "wb") as f:
    pickle.dump(account_to_idx, f)


In [ ]:
def build_graph(edge_subset, eval_mask):
    src = edge_subset['src_account'].map(account_to_idx).values
    dst = edge_subset['dst_account'].map(account_to_idx).values
    edge_index = torch.tensor(np.stack([src, dst], axis=0), dtype=torch.long)

    edge_attr = torch.tensor(
        edge_subset[EDGE_FEAT_COLS].values.astype(np.float32), dtype=torch.float)

    edge_time = torch.tensor(
        edge_subset['Timestamp'].astype(np.int64).values // 10**9, dtype=torch.long)

    labels = np.full(len(edge_subset), -1, dtype=np.int64)
    labels[eval_mask] = edge_subset.loc[eval_mask, 'label'].values.astype(np.int64)
    y = torch.tensor(labels, dtype=torch.long)

    return Data(
        x          = X_node,
        edge_index = edge_index,
        edge_attr  = edge_attr,
        edge_time  = edge_time,
        y          = y,
        eval_mask  = torch.tensor(eval_mask, dtype=torch.bool),
        num_nodes  = N_nodes,
    )


In [ ]:
print('Building graph snapshots ...')

with Timer('train graph'):
    train_edges = edge_df[train_mask].reset_index(drop=True)
    train_graph = build_graph(train_edges, np.ones(len(train_edges), dtype=bool))

with Timer('val graph'):
    val_df   = edge_df[train_mask | val_mask].reset_index(drop=True)
    val_eval = np.zeros(len(val_df), dtype=bool); val_eval[t1_idx:] = True
    val_graph = build_graph(val_df, val_eval)

with Timer('test graph'):
    all_df    = edge_df.reset_index(drop=True)
    test_eval = np.zeros(len(all_df), dtype=bool); test_eval[t2_idx:] = True
    test_graph = build_graph(all_df, test_eval)

print('All snapshots built ✓')

In [ ]:
def summarise(name, g):
    n_eval  = g.eval_mask.sum().item()
    n_laund = (g.y[g.eval_mask] == 1).sum().item()
    rate    = n_laund / n_eval * 100 if n_eval > 0 else 0
    print(f'  {name:<12} | nodes={g.num_nodes:>7,} | edges={g.edge_index.shape[1]:>9,} '
          f'| eval={n_eval:>9,} | laund={n_laund:>5,} ({rate:.4f}%)')

print('\n── Graph snapshots ──────────────────────────────────────────────────────')
summarise('train_graph', train_graph)
summarise('val_graph',   val_graph)
summarise('test_graph',  test_graph)
print(f'\nEdge feature dim : {train_graph.edge_attr.shape[1]}')
print(f'Node feature dim : {train_graph.x.shape[1]}')
print()
print('════════════════════════════════════════════════════════════')
print(f'  ACTION: set  EDGE_DIM = {train_graph.edge_attr.shape[1]}  in baseline_model.ipynb')
print('════════════════════════════════════════════════════════════')

In [ ]:
torch.save(train_graph, 'Data/train_graph_enhanced_v2.pt')
torch.save(val_graph,   'Data/val_graph_enhanced_v2.pt')
torch.save(test_graph,  'Data/test_graph_enhanced_v2.pt')

g = torch.load('Data/train_graph_enhanced_v2.pt', weights_only=False)
assert g.edge_attr.shape[1] == len(EDGE_FEAT_COLS), 'Feature dim mismatch!'

print('Saved:')
print('  Data/train_graph_enhanced_v2.pt')
print('  Data/val_graph_enhanced_v2.pt')
print('  Data/test_graph_enhanced_v2.pt')
print(f'\nReload check -> edges: {g.edge_index.shape[1]:,}  edge_attr: {g.edge_attr.shape}  OK')


---
## Appendix — Sanity checks

In [ ]:
# ── Check 1: every account's first transaction must have all history = 0 ─────
first_mask = edges.groupby('src_account').cumcount() == 0

check_cols = [
    'src_out_count_life', 'src_out_vol_1d',
    'src_in_count_life',  'src_in_vol_1d',
    'dst_out_count_life', 'dst_out_vol_1d',
    'src_fanout_uniq_6h', 'closes_cycle',
    'src_cycle_count',    'dst_cycle_count',
]
result   = edges.loc[first_mask, check_cols]
all_zero = (result == 0).all().all()

print('Check 1 — History features are 0 on first transaction (leakage guard):')
print(result.describe().loc[['max']].round(4))
print(f'\nAll zero: {all_zero}  {"✓" if all_zero else "✗ LEAKAGE DETECTED"}')

In [ ]:
# ── Check 2: laundering vs legitimate feature means ───────────────────────────
# Ratio > 1 means laundering transactions show higher values — good signal.
signal_cols = [
    'src_out_spike_1d',    'src_out_diversity_ratio',
    'src_in_vol_1d',       'src_in_unique_life',
    'dst_in_spike_1d',     'dst_in_diversity_ratio',
    'dst_out_vol_1d',      'dst_resend_ratio_1d',
    'src_fanout_uniq_6h',  'dst_fanin_uniq_6h',  'sg_combined_6h',
    'closes_cycle',        'src_cycle_count',     'dst_cycle_count',
]

legit = edges[edges['label'] == 0][signal_cols]
laund = edges[edges['label'] == 1][signal_cols]

comp = pd.DataFrame({
    'Legit mean'        : legit.mean(),
    'Laundering mean'   : laund.mean(),
    'Ratio laund/legit' : (laund.mean() / (legit.mean() + 1e-9)).round(2),
})

print('Check 2 — Mean feature values: laundering vs legitimate transactions')
print('Ratio > 1 = laundering edges show higher values = discriminative signal')
print()
print(comp.round(4).to_string())